# Td3 Vehicle Navigation

Custom grid observations, CNN features and a TD3 agent for a simulated navigation task.

Stage: prototype.


In [ ]:
!pip install stable-baselines3
!pip install 'shimmy>=0.2.1'
!pip install gymnasium[all]



# Essential Libraries for RL and Deep Learning Models
import gym
from gym import spaces
import numpy as np
from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import matplotlib.pyplot as plt
from tqdm import tqdm

# Custom CNN for Feature Extraction from Grid-based Inputs
class VehicleCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, output_dim=256, **kwargs):
        super(VehicleCNN, self).__init__(observation_space, output_dim)
        # Define convolutional layers with pooling and a dense layer for flattened output
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten()
        )
        # Calculate number of features and connect to output dense layer
        with torch.no_grad():
            n_flattened = self.cnn_layers(torch.zeros(1, *observation_space.shape)).shape[1]
        self.fc_layer = nn.Sequential(nn.Linear(n_flattened, output_dim), nn.ReLU())

    def forward(self, inputs):
        return self.fc_layer(self.cnn_layers(inputs))

# Enhanced 2D Environment for Autonomous Vehicle Simulation
class AVEnvironment(gym.Env):
    def __init__(self, path_type="circular"):
        super(AVEnvironment, self).__init__()
        self.path_type = path_type
        self.action_space = spaces.Box(low=np.array([-1, -1]), high=np.array([1, 1]), dtype=np.float32)
        self.observation_space = spaces.Box(low=0, high=1, shape=(1, 10, 10), dtype=np.float32)
        self.last_action = np.array([0, 0])
        self.time_step = 0
        self.reset()

    def reset(self):
        # State: [x, y, speed, heading, dist_to_waypoint, dist_to_obstacle]
        self.state = np.array([0.0, 0.0, 1.0, 0.0, 0.0, 0.0])
        self.goal = self._define_circular_goal()
        self.obstacles = np.random.uniform(-5, 15, size=(5, 2))
        self.time_step = 0
        self.last_action = np.array([0, 0])
        # Calculate initial distances
        self.state[4] = np.linalg.norm(self.state[:2] - self.goal)
        self.state[5] = min(np.linalg.norm(self.state[:2] - obs) for obs in self.obstacles)

        return self._generate_observation_grid()

    def step(self, action):
        # Reward penalty based on abrupt changes in actions (jerk)
        jerk_penalty = np.sum(np.abs(action - self.last_action))
        self.last_action = action
        steering, acceleration = action
        x, y, speed, heading, _, _ = self.state
        speed = np.clip(speed + acceleration * 0.1, 0, 5)
        heading += steering * 0.1
        x += speed * np.cos(heading)
        y += speed * np.sin(heading)

        # Recalculate distances
        dist_to_waypoint = np.linalg.norm(np.array([x, y]) - self.goal)
        dist_to_obstacle = min(np.linalg.norm(np.array([x, y]) - obs) for obs in self.obstacles)

        # Update state
        self.state = np.array([x, y, speed, heading, dist_to_waypoint, dist_to_obstacle])

        # Smoothness-based reward with added goal proximity bonus
        reward = self._reward_function(dist_to_waypoint, dist_to_obstacle, jerk_penalty)
        for obs in self.obstacles:
            obs += np.random.uniform(-0.2, 0.2, size=obs.shape)  # Dynamic obstacles

        done = dist_to_waypoint < 1.0 or dist_to_obstacle < 1.5 or self.time_step > 200
        self.time_step += 1
        return self._generate_observation_grid(), reward, done, {}

    def _reward_function(self, dist_to_waypoint, dist_to_obstacle, jerk_penalty):
        # Lower jerk penalty, added goal proximity bonus
        reward = -0.3 * jerk_penalty - dist_to_waypoint * 0.02

        # Significant penalty if near-collision
        if dist_to_obstacle < 1.5:
            reward -= 10

        # Large reward on goal reach
        if dist_to_waypoint < 1.0:
            reward += 500
        elif dist_to_waypoint < 2.0:
            reward += 100

        return reward

    def _define_circular_goal(self):
        angle = (self.time_step / 200) * 2 * np.pi
        return np.array([10.0 * np.cos(angle), 10.0 * np.sin(angle)])

    def _generate_observation_grid(self):
        x, y, _, _, _, _ = self.state
        grid = np.zeros((10, 10))

        goal_rel = ((self.goal - [x, y]) + 5).astype(int)
        if 0 <= goal_rel[0] < 10 and 0 <= goal_rel[1] < 10:
            grid[goal_rel[0], goal_rel[1]] = 1

        for obs in self.obstacles:
            obs_rel = ((obs - [x, y]) + 5).astype(int)
            if 0 <= obs_rel[0] < 10 and 0 <= obs_rel[1] < 10:
                grid[obs_rel[0], obs_rel[1]] = 0.5

        return grid.reshape((1, 10, 10))

# Initialize environment and TD3 agent
env = AVEnvironment()
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_td3 = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1, device=device)

# Agent evaluation with key metrics tracking
def evaluate_agent(env, model, num_episodes=10):
    metrics = {"avg_reward": [], "avg_path_deviation": [], "avg_collisions": [], "avg_smoothness": []}
    for episode in range(num_episodes):
        state = env.reset()
        episode_reward, deviation, collision_count, smoothness_score = 0, 0, 0, 0
        done = False
        last_action = np.array([0, 0])

        while not done:
            action, _ = model.predict(state, deterministic=True)
            jerk_penalty = np.sum(np.abs(action - last_action))
            smoothness_score += jerk_penalty
            last_action = action
            next_state, reward, done, _ = env.step(action)
            deviation += env.state[4]
            collision_count += int(env.state[5] < 1.5)
            episode_reward += reward
            state = next_state

        metrics["avg_reward"].append(episode_reward)
        metrics["avg_path_deviation"].append(deviation)
        metrics["avg_collisions"].append(collision_count)
        metrics["avg_smoothness"].append(smoothness_score)

    return {key: np.mean(value) for key, value in metrics.items()}

# Training loop with periodic evaluation
total_timesteps, eval_step = 500000, 50000
results = {"rewards": [], "collisions": [], "deviations": []}

with tqdm(total=total_timesteps) as pbar:
    for i in range(total_timesteps // eval_step):
        model_td3.learn(total_timesteps=eval_step, reset_num_timesteps=False, log_interval=200)
        metrics = evaluate_agent(env, model_td3, num_episodes=10)
        print(f"Evaluation {i + 1} - Metrics: {metrics}")
        results["rewards"].append(metrics["avg_reward"])
        results["collisions"].append(metrics["avg_collisions"])
        results["deviations"].append(metrics["avg_path_deviation"])
        pbar.update(eval_step)

# Plot training metrics for visual insights
plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
plt.plot(results["rewards"], label="Reward")
plt.xlabel("Eval Step")
plt.ylabel("Avg Reward")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(results["collisions"], label="Collisions", color='orange')
plt.xlabel("Eval Step")
plt.ylabel("Avg Collisions")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(results["deviations"], label="Path Deviation", color='green')
plt.xlabel("Eval Step")
plt.ylabel("Avg Path Deviation")
plt.legend()

plt.tight_layout()
plt.show()

# Final evaluation
final_metrics = evaluate_agent(env, model_td3, num_episodes=50)
print("Final Evaluation Metrics:", final_metrics)

# Save the TD3 model
model_td3.save("td3_AV_navigation_trained")
print("Model saved.")
